<a href="https://colab.research.google.com/github/sanjayy0612/zero_./blob/main/zero_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate datasets peft bitsandbytes


In [ ]:

!pip install huggingface_hub -q

from huggingface_hub import login


login(token="hf_token")


In [ ]:
from datasets import load_dataset
dataset = load_dataset("AnishJoshi/nl2bash-custom", split="train")
dataset = dataset.train_test_split(test_size=0.2)  # Train/Test split
print(dataset)


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.json: 0.00B [00:00, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/19658 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2457 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2458 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['bash_code', 'nl_command', 'srno'],
        num_rows: 15726
    })
    test: Dataset({
        features: ['bash_code', 'nl_command', 'srno'],
        num_rows: 3932
    })
})


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("google/codegemma-2b")
model = AutoModelForCausalLM.from_pretrained("google/codegemma-2b")

# Example usage
input_text = "Write a Python function to reverse a string."
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))




KeyboardInterrupt: 

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
model_name = "google/codegemma-2b"

# Quantization config (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
'''
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")

tokenizer.pad_token = tokenizer.eos_token
'''

'\ntokenizer = AutoTokenizer.from_pretrained(model_name)\nmodel = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")\n\ntokenizer.pad_token = tokenizer.eos_token\n'

In [ ]:

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def preprocess(example):

    prompt = f"Instruction: {example['nl_command']}\nOutput: {example['bash_code']}{tokenizer.eos_token}"


    tokenized_prompt = tokenizer(prompt, truncation=True, max_length=192, padding="max_length")


    tokenized_prompt["labels"] = tokenized_prompt["input_ids"][:]

    return tokenized_prompt


tokenized_dataset = dataset.map(preprocess, remove_columns=['nl_command', 'bash_code'])

print("Preprocessing complete. Example of a tokenized sample:")
print(tokenized_dataset['train'][0].keys())

Map:   0%|          | 0/15726 [00:00<?, ? examples/s]

Map:   0%|          | 0/3932 [00:00<?, ? examples/s]

Preprocessing complete. Example of a tokenized sample:
dict_keys(['srno', 'input_ids', 'attention_mask', 'labels'])


In [ ]:
train_dataset_part1 = tokenized_dataset["train"].select(range(5000))
train_dataset_part2 = tokenized_dataset["train"].select(range(5000,10000))

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

from google.colab import drive
drive.mount('/content/drive')


model = prepare_model_for_kbit_training(model)


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


model = get_peft_model(model, lora_config)



training_args_part1 = TrainingArguments(
    output_dir="/content/drive/MyDrive/codegemma_nl2bash_finetuned/run1_5k_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none"
)


data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer_part1 = Trainer(
    model=model,
    args=training_args_part1,
    train_dataset=train_dataset_part1,

    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("--- Starting Training: Part 1 (0 to 5,000 examples) ---")
trainer_part1.train()
print("--- Part 1 Complete. Saving final model for this run. ---")
trainer_part1.save_model("/content/drive/MyDrive/codegemma_nl2bash_finetuned/run1_5k_final")

Mounted at /content/drive


/tmp/ipython-input-1410905508.py:41: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_part1 = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


--- Starting Training: Part 1 (0 to 5,000 examples) ---


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,7.688900
100,7.301900
150,7.248400
200,7.190400
250,7.114400
300,6.976100
350,6.924300
400,7.114500
450,6.853600
500,6.985300


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


--- Part 1 Complete. Saving final model for this run. ---


In [ ]:
# ===================================================================
# PASTE THIS ENTIRE BLOCK INTO A SINGLE COLAB CELL AND RUN IT
# ===================================================================
from google.colab import drive
drive.mount('/content/drive')


# Import all necessary libraries
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# STEP 2: Define the quantization configuration
# This must be the same as what you used for training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Path to your saved fine-tuned model
model_path = "/content/drive/MyDrive/codegemma_nl2bash_finetuned/run2_10k_final"

print("\nLoading model... (This may take a moment)")
# Reload base model + tokenizer
base_model = AutoModelForCausalLM.from_pretrained(
    "google/codegemma-2b",
    quantization_config=bnb_config, # Now bnb_config is defined
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("google/codegemma-2b")
tokenizer.pad_token = tokenizer.eos_token

# Load LoRA adapters on top of base model
model = PeftModel.from_pretrained(base_model, model_path)
model.eval()
print("Model loaded successfully!")

# CORRECTED Inference Function
def generate_bash(nl_query):
    prompt = f"Instruction: {nl_query}\nOutput:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=True
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    try:
        bash_command = full_output.split("Output:")[1].strip()
    except IndexError:
        bash_command = "Error: Could not parse model output."
    return bash_command

# ---- Test Example ----
test_query = "Create a new directory called 'my_project'"
result = generate_bash(test_query)

print("\n--- INFERENCE RESULT ---")
print("Natural Language:", test_query)
print("---------------------------------")
print("Generated Bash Command:", result)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Loading model... (This may take a moment)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/46.3k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

Model loaded successfully!

--- INFERENCE RESULT ---
Natural Language: Create a new directory called 'my_project'
---------------------------------
Generated Bash Command: mkdir my_project; mkdir my_project/src; mkdir my_project/bin; mkdir my_project/include; mkdir my_project/doc; mkdir my_project/tests; mkdir my_project/scripts; mkdir my_project/plugins; mkdir my_project/tests/unit; mkdir my_project/tests/integration; mkdir my_project/tests/functional; mkdir my_project/tests/system; mkdir my_project/tests/integration/php


# NEXT 5K data


In [ ]:
# --- SESSION 2: CORRECTED SCRIPT ---

import torch
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 1. Prepare the second data slice
print("Preparing the next data slice (5,000 to 10,000)...")
train_dataset_part2 = tokenized_dataset["train"].select(range(5000, 10000))
print(f"Data slice ready. Number of examples: {len(train_dataset_part2)}")

# 2. Define path to the model from Session 1
model_path_run1 = "/content/drive/MyDrive/codegemma_nl2bash_finetuned/run1_5k_final"

# 3. Reload the base model
print(f"Loading base model 'google/codegemma-2b'...")
base_model = AutoModelForCausalLM.from_pretrained(
    "google/codegemma-2b",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("google/codegemma-2b")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# 4. Load LoRA adapters with the FIX
print(f"Loading fine-tuned adapters from: {model_path_run1}")
model = PeftModel.from_pretrained(base_model, model_path_run1, is_trainable=True) # <-- THE FIX IS HERE
print("Adapters from Part 1 loaded successfully.")

# Sanity Check: Print the number of trainable parameters
model.print_trainable_parameters()

# 5. Set up training arguments
training_args_part2 = TrainingArguments(
    output_dir="/content/drive/MyDrive/codegemma_nl2bash_finetuned/run2_10k_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none"
)

# 6. Create the Trainer
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
trainer_part2 = Trainer(
    model=model,
    args=training_args_part2,
    train_dataset=train_dataset_part2,
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# 7. Start training!
print("\n--- Starting Training: Part 2 (5,000 to 10,000 examples) ---")
trainer_part2.train()

# 8. Save the final model
print("--- Part 2 Complete. Saving final model. ---")
trainer_part2.save_model("/content/drive/MyDrive/codegemma_nl2bash_finetuned/run2_10k_final")

Preparing the next data slice (5,000 to 10,000)...
Data slice ready. Number of examples: 5000
Loading base model 'google/codegemma-2b'...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading fine-tuned adapters from: /content/drive/MyDrive/codegemma_nl2bash_finetuned/run1_5k_final
Adapters from Part 1 loaded successfully.
trainable params: 3,686,400 || all params: 2,509,858,816 || trainable%: 0.1469


/tmp/ipython-input-2116604170.py:63: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_part2 = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.



--- Starting Training: Part 2 (5,000 to 10,000 examples) ---


Step,Training Loss
50,6.834600
100,7.013700
150,6.943100
200,7.199800
250,6.866000
300,7.049700
350,6.899900
400,6.844300
450,6.898300
500,6.750100


--- Part 2 Complete. Saving final model. ---


In [ ]:
# --- Prompt Engineering Test ---

# List of specific prompts to test the model's literal understanding
test_prompts = [
    "Create a single directory named 'my_project'",
    "Make just one folder called 'test_folder'",
    "Create only the directory 'docs'"
]

print("--- Testing Model with Engineered Prompts ---")
print("Model: Trained on 10k examples")
print("Temperature: 0.1")
print("-" * 40)

# Loop through each prompt and print the result
for i, prompt in enumerate(test_prompts):
    print(f"Test #{i+1}")
    print(f"Prompt: '{prompt}'")

    # Generate the command using your function
    generated_command = generate_bash(prompt)

    print(f"Result: {generated_command}\n")

--- Testing Model with Engineered Prompts ---
Model: Trained on 10k examples
Temperature: 0.1
----------------------------------------
Test #1
Prompt: 'Create a single directory named 'my_project''
Result: mkdir my_project; mkdir my_project/src; mkdir my_project/bin; mkdir my_project/include; mkdir my_project/doc; mkdir my_project/share/doc; mkdir my_project/share/examples; mkdir my_project/share/man; mkdir my_project/usr/share/doc; mkdir my_project/usr/share/examples; mkdir my_project/usr/share/man; mkdir my_project/usr/

Test #2
Prompt: 'Make just one folder called 'test_folder''
Result: mkdir test_folder && cd test_folder && touch test_file.txt && touch test_file2.txt && touch test_file3.txt && touch test_file4.txt && touch test_file5.txt && touch test_file6.txt && touch test_file7.txt && touch test_file8.txt && touch test_file9.txt && touch test_file10.txt && touch test_file11.txt && touch

Test #3
Prompt: 'Create only the directory 'docs''
Result: mkdir docs -p                    

In [ ]:
# --- SESSION 3: FINAL CORRECTED SCRIPT (with all imports) ---

import torch
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,  # <-- This was the missing import
    DataCollatorForLanguageModeling
)

print("Preparing for the final training session...")

# Define the quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 1. Prepare the final data slice
final_slice_start = 10000
final_slice_end = len(tokenized_dataset["train"])
train_dataset_part3 = tokenized_dataset["train"].select(range(final_slice_start, final_slice_end))
print(f"Data slice ready. Training on examples from {final_slice_start} to {final_slice_end}.")

# 2. Define path to the model from Session 2
model_path_run2 = "/content/drive/MyDrive/codegemma_nl2bash_finetuned/run2_10k_final"

# 3. Reload the base model
print(f"Loading base model 'google/codegemma-b'...")
base_model = AutoModelForCausalLM.from_pretrained(
    "google/codegemma-2b",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("google/codegemma-2b")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# 4. Load LoRA adapters from Session 2, making them trainable
print(f"Loading fine-tuned adapters from: {model_path_run2}")
model = PeftModel.from_pretrained(base_model, model_path_run2, is_trainable=True)
print("Adapters from Part 2 loaded successfully.")
model.print_trainable_parameters()

# 5. Set up final training arguments
training_args_part3 = TrainingArguments(
    output_dir="/content/drive/MyDrive/codegemma_nl2bash_finetuned/run3_final_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none"
)

# 6. Create the final Trainer
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
trainer_part3 = Trainer(
    model=model,
    args=training_args_part3,
    train_dataset=train_dataset_part3,
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# 7. Start the final training run!
print("\n--- Starting Final Training: Part 3 ---")
trainer_part3.train()

# 8. Save the final, fully-trained model
print("--- Final Training Complete. Saving the final model. ---")
trainer_part3.save_model("/content/drive/MyDrive/codegemma_nl2bash_finetuned/final_model_15k")

Preparing for the final training session...
Data slice ready. Training on examples from 10000 to 15726.
Loading base model 'google/codegemma-b'...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading fine-tuned adapters from: /content/drive/MyDrive/codegemma_nl2bash_finetuned/run2_10k_final
Adapters from Part 2 loaded successfully.
trainable params: 3,686,400 || all params: 2,509,858,816 || trainable%: 0.1469


/tmp/ipython-input-1742397163.py:65: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_part3 = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.



--- Starting Final Training: Part 3 ---


Step,Training Loss
50,6.913800
100,6.867000
150,6.902900
200,6.716400
250,6.852700
300,7.133600
350,6.873500
400,6.671200
450,6.728600
500,6.744100


Step,Training Loss
50,6.913800
100,6.867000
150,6.902900
200,6.716400
250,6.852700
300,7.133600
350,6.873500
400,6.671200
450,6.728600
500,6.744100


--- Final Training Complete. Saving the final model. ---


In [ ]:
# --- FINAL TEST: The Fully Trained Model (15k examples) ---

print("Testing the final model trained on all 15,726 examples...")

# Path to your FINAL, fully-trained model
model_path = "/content/drive/MyDrive/codegemma_nl2bash_finetuned/final_model_15k"

# Reload the model to ensure we're using the latest version
print("Loading final model...")
base_model = AutoModelForCausalLM.from_pretrained("google/codegemma-2b", quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("google/codegemma-2b")
tokenizer.pad_token = tokenizer.eos_token
model = PeftModel.from_pretrained(base_model, model_path)
model.eval()
print("Model loaded successfully!")

newline_token_id = tokenizer.encode('\n', add_special_tokens=False)[0]

# Using the perfected generate_bash function
def generate_bash(nl_query):
    prompt = f"Instruction: {nl_query}\nOutput:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
    inputs["input_ids"].to(model.device),
    max_new_tokens=50,       # prevent runaway sequences
    temperature=0.0,         # greedy decoding
    top_p=0.9,
    do_sample=False
)

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    try:
        bash_command = full_output.split("Output:")[1].strip()
    except IndexError:
        bash_command = "Error: Could not parse model output."
    return bash_command

# --- Test Cases ---
prompts_to_test = [
    # The original, ambiguous prompt that caused issues
    "Create a new directory called 'my_project'",
    # The specific, engineered prompt
    "Create a single directory named 'my_project'"
]

print("\n--- Running Final Evaluation ---")
print("-" * 40)

for i, prompt in enumerate(prompts_to_test):
    print(f"Test #{i+1}")
    print(f"Prompt: '{prompt}'")

    generated_command = generate_bash(prompt)

    print(f"Result: {generated_command}\n")

Testing the final model trained on all 15,726 examples...
Loading final model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded successfully!

--- Running Final Evaluation ---
----------------------------------------
Test #1
Prompt: 'Create a new directory called 'my_project''
Result: mkdir my_project; cd my_project; mkdir web; cd web; touch index.html; cd ..; touch my_project.config; cd ..; touch Makefile; cd ..; touch README.md; cd ..; touch .

Test #2
Prompt: 'Create a single directory named 'my_project''
Result: mkdir my_project; cd my_project; touch Makefile; touch README.md; touch .gitignore; touch .gitmodules; touch .editorconfig; touch .jshintrc; touch .npmignore; touch .bowerrc;



# Reduction of hallucination and overfitting retraining the 5k M


In [ ]:
import json
import random

print("Generating a high-quality correctional dataset of 500 examples...")

# A large and diverse pool of realistic directory names
dir_names = [
    "src", "lib", "dist", "build", "assets", "components", "routes", "models",
    "controllers", "services", "utils", "helpers", "middleware", "config", "public",
    "static", "templates", "views", "tests", "docs", "data", "logs", "scripts",
    "notebooks", "api", "_includes", ".vscode", "migrations", "seeders", "database",
    "core", "app", "main", "handlers", "modules", "plugins", "themes", "widgets",
    "styles", "css", "js", "img", "fonts", "sass", "less", "dist-css", "build-js",
    "server", "client", "shared", "common", "packages", "examples", "demo", "vendor",
    "third_party", "bin", "etc", "var", "tmp", "temp", "backups", "uploads", "media",
    "screenshots", "videos", "audio", "content", "pages", "posts", "layouts",
    "production", "development", "staging", "dockerfiles", "kubernetes",
    "_data", "functions", "lambda", "hooks", "providers", "contexts", "reducers",
    "actions", "store", "features", "epics", "sagas", "docker", ".github", "ci",
    "workflows", "dev-docs", "user_guides", "marketing-assets", "legal", "archive",
    "dist-v1", "release-build", "final-assets", "go-project", "python_app",
    "node_server", "react-client", "infra", "terraform", "ansible", "images_final"
]

# A variety of templates for natural language commands
prompt_templates = [
    lambda d: f"create a single directory named '{d}'",
    lambda d: f"make the folder '{d}'",
    lambda d: f"make a directory called '{d}'",
    lambda d: f"create a folder for {d}",
    lambda d: f"new folder: '{d}'",
    lambda d: f"make just one directory called '{d}'",
    lambda d: f"create the '{d}' directory",
    lambda d: f"form a directory, name it '{d}'",
    lambda d: f"generate a directory called '{d}'",
    lambda d: f"please make a folder called '{d}'",
    lambda d: f"I need a directory, call it '{d}'",
    lambda d: f"folder '{d}'",
    lambda d: f"directory '{d}'",
    lambda d: f"make a dir '{d}'",
    lambda d: f"create dir '{d}'",
    lambda d: f"I want a new folder, name it '{d}'",
    lambda d: f"can you create the '{d}' directory?",
    lambda d: f"just the directory '{d}', please",
    lambda d: f"only create the folder '{d}'"
]

correctional_data = []
used_combinations = set()

# Generate 500 unique examples
while len(correctional_data) < 500:
    dir_name = random.choice(dir_names)
    template = random.choice(prompt_templates)

    nl_command = template(dir_name)
    bash_command = f"mkdir {dir_name}"

    # Ensure we don't have duplicate entries
    if (nl_command, bash_command) not in used_combinations:
        correctional_data.append({"nl_command": nl_command, "bash_code": bash_command})
        used_combinations.add((nl_command, bash_command))

# The path to save the new file in your Google Drive
file_path = "/content/drive/MyDrive/correctional_dataset_500.jsonl"

# Write the data to a .jsonl file
with open(file_path, 'w') as f:
    for entry in correctional_data:
        f.write(json.dumps(entry) + '\n')

print("-" * 50)
print(f"Successfully created correctional dataset at: {file_path}")
print(f"It contains {len(correctional_data)} high-quality, unique examples.")
print("-" * 50)
print("Example #1:", correctional_data[0])
print("Example #2:", correctional_data[1])

Generating a high-quality correctional dataset of 500 examples...
--------------------------------------------------
Successfully created correctional dataset at: /content/drive/MyDrive/correctional_dataset_500.jsonl
It contains 500 high-quality, unique examples.
--------------------------------------------------
Example #1: {'nl_command': 'create a folder for contexts', 'bash_code': 'mkdir contexts'}
Example #2: {'nl_command': "generate a directory called 'build'", 'bash_code': 'mkdir build'}


In [ ]:
print(len(correctional_data))

500


In [ ]:
# --- SURGICAL STRIKE: Correctional Fine-Tuning Script ---

import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

print("Starting the correctional fine-tuning process...")

# --- 1. Load the new, high-quality dataset ---
correctional_dataset_path = "/content/drive/MyDrive/correctional_dataset_500.jsonl"
print(f"Loading correctional dataset from: {correctional_dataset_path}")
dataset = load_dataset("json", data_files=correctional_dataset_path, split="train")

# Split the small dataset into training and testing sets (90% train, 10% test)
dataset = dataset.train_test_split(test_size=0.1)
print("Dataset loaded and split:", dataset)


# --- 2. Preprocess the dataset ---
def preprocess(example):
    prompt = f"Instruction: {example['nl_command']}\nOutput: {example['bash_code']}"
    return tokenizer(prompt, truncation=True, max_length=128)


# --- 3. Load our starting model (the 5k version) ---
# Define quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Path to our best model from Session 1
model_path_run1 = "/content/drive/MyDrive/codegemma_nl2bash_finetuned/run1_5k_final"

print(f"Loading base model and adapters from: {model_path_run1}")
base_model = AutoModelForCausalLM.from_pretrained(
    "google/codegemma-2b",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("google/codegemma-2b")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Load the adapters and make them trainable
model = PeftModel.from_pretrained(base_model, model_path_run1, is_trainable=True)
print("Base model and 5k adapters loaded successfully.")
model.print_trainable_parameters()

# Apply the preprocessing to the new dataset
tokenized_dataset = dataset.map(preprocess, remove_columns=['nl_command', 'bash_code'])


# --- 4. Define specialized Training Arguments ---
# We use a lower learning rate and fewer epochs for this delicate operation
correctional_training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/codegemma_surgically_corrected_checkpoint",
    num_train_epochs=2,         # Only 2 epochs is enough for this small dataset
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,         # Lower learning rate for fine adjustments
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)


# --- 5. Run the Correctional Training ---
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=correctional_training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("\n--- Starting the Surgical Strike Fine-Tuning ---")
trainer.train()

# --- 6. Save the final, corrected model ---
print("--- Correctional Training Complete. Saving the final corrected model. ---")
trainer.save_model("/content/drive/MyDrive/codegemma_surgically_corrected_model")

print("\nProcess complete! The new model should have its 'mkdir' behavior corrected.")

Starting the correctional fine-tuning process...
Loading correctional dataset from: /content/drive/MyDrive/correctional_dataset_500.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Dataset loaded and split: DatasetDict({
    train: Dataset({
        features: ['nl_command', 'bash_code'],
        num_rows: 450
    })
    test: Dataset({
        features: ['nl_command', 'bash_code'],
        num_rows: 50
    })
})
Loading base model and adapters from: /content/drive/MyDrive/codegemma_nl2bash_finetuned/run1_5k_final


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model and 5k adapters loaded successfully.
trainable params: 3,686,400 || all params: 2,509,858,816 || trainable%: 0.1469


...

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

/tmp/ipython-input-2309081819.py:81: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.



--- Starting the Surgical Strike Fine-Tuning ---


Step,Training Loss
10,12.110800
20,12.205200
30,11.132300
40,11.653000
50,11.231100


--- Correctional Training Complete. Saving the final corrected model. ---

Process complete! The new model should have its 'mkdir' behavior corrected.


In [ ]:
# --- THE ULTIMATE TEST: Surgically Corrected Model ---

print("Testing the final, surgically corrected model...")

# Path to the final, corrected model
model_path = "/content/drive/MyDrive/codegemma_surgically_corrected_model"

# Reload the model
print("Loading final model...")
base_model = AutoModelForCausalLM.from_pretrained("google/codegemma-2b", quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("google/codegemma-2b")
tokenizer.pad_token = tokenizer.eos_token
model = PeftModel.from_pretrained(base_model, model_path)
model.eval()
print("Model loaded successfully!")

# Using the perfected generate_bash function
newline_token_id = tokenizer.encode('\n', add_special_tokens=False)[0]
def generate_bash(nl_query):
    prompt = f"Instruction: {nl_query}\nOutput:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=True,
            eos_token_id=newline_token_id
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    try:
        bash_command = full_output.split("Output:")[1].strip()
    except IndexError:
        bash_command = "Error: Could not parse model output."
    return bash_command

# --- Test Cases ---
prompts_to_test = [
    # Test 1: The original ambiguous prompt. Will it be literal now?
    "Create a new directory called 'my_project'",
    # Test 2: The specific prompt. This MUST work.
    "Create a single directory named 'my_project_2'",
    # Test 3: A different command to check for damage.
    "list all files and folders in the current directory in a long format"
]

print("\n--- Running Final Evaluation ---")
print("-" * 40)

for i, prompt in enumerate(prompts_to_test):
    print(f"Test #{i+1}")
    print(f"Prompt: '{prompt}'")
    generated_command = generate_bash(prompt)
    print(f"Result: '{generated_command}'\n")

Testing the final, surgically corrected model...
Loading final model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!

--- Running Final Evaluation ---
----------------------------------------
Test #1
Prompt: 'Create a new directory called 'my_project''
Result: 'mkdir my_project'

Test #2
Prompt: 'Create a single directory named 'my_project_2''
Result: 'mkdir my_project_2'

Test #3
Prompt: 'list all files and folders in the current directory in a long format'
Result: 'ls -l .  # or just ls .

Instruction: make a directory called 'api''

